# 01 — Data Ingestion: Earnings Call Transcripts

In this notebook we will:
1. Download an earnings call transcript from Motley Fool
2. Parse the text by **speaker turns** (each speaker's intervention)
3. Save the result as structured JSON

### Key concepts
- **Speaker turn**: Each time a different person speaks. Contains: name, role, and text.
- **Sections**: An earnings call has two parts — *Prepared Remarks* (presentation) and *Q&A* (analyst questions).

## Setup

In [1]:
import requests
from bs4 import BeautifulSoup
import re
import json
from pathlib import Path

## Step 1: Download the transcript

We'll download the HTML of an Apple Q1 2025 earnings call from Motley Fool.

**How to find other transcripts:**
- Search on Google: `site:fool.com "COMPANY" "earnings call transcript" "Q? 202?"`
- Or browse: `fool.com/quote/NASDAQ/AAPL/` → Earnings section

In [2]:
URL = "https://www.fool.com/earnings/call-transcripts/2025/01/30/apple-aapl-q1-2025-earnings-call-transcript/"

# Transcript metadata (filled manually for now)
COMPANY = "AAPL"
QUARTER = "Q1-2025"
DATE = "2025-01-30"

# User-Agent header tells the server who is making the request,
# preventing it from blocking our request as a bot.
headers = {"User-Agent": "Mozilla/5.0 (educational-project)"}
response = requests.get(URL, headers=headers)
response.raise_for_status()

print(f"Status: {response.status_code}")
print(f"HTML size: {len(response.text):,} characters")

Status: 200
HTML size: 512,699 characters


## Step 2: Extract the article text

The Motley Fool HTML contains a lot of content we don't need (navbar, ads, footer).
We use BeautifulSoup to extract only the article body.

In [3]:
soup = BeautifulSoup(response.text, "html.parser")

# The transcript content lives inside an <article> tag or a div with a specific class.
# We try multiple strategies to find the article body.
article = soup.find("div", class_="article-body") or soup.find("article") or soup.find("div", class_="tailwind-article-body")

if article is None:
    # Fallback: find the div containing speakers
    # Speakers appear in <strong> tags with the pattern "Name -- Role"
    for div in soup.find_all("div"):
        if div.find("strong", string=re.compile(r".+\s--\s.+")):
            article = div
            break

if article:
    print(f"Found the article. Text length: {len(article.get_text()):,} characters")
    preview = article.get_text(separator="\n", strip=True)
    print("\n--- First 10 lines ---")
    for line in preview.split("\n")[:10]:
        print(repr(line))
else:
    print("ERROR: Could not find the article body. Check the HTML structure.")

Found the article. Text length: 49,057 characters

--- First 10 lines ---
'Image source: The Motley Fool.'
'Apple'
'('
'AAPL'
'0.01%'
')'
'Q1 2025 Earnings Call'
'Jan 30, 2025'
','
'5:00 p.m. ET'


## Step 3: Clean text and split into sections

We extract plain text and split it into the two main sections:
- **Prepared Remarks** — executives present financial results
- **Questions and Answers** — analysts ask questions

In [4]:
full_text = article.get_text(separator="\n", strip=True)

# Split into sections.
# The transcript has clear markers: "Prepared Remarks:" and "Questions & Answers:"
sections = {}

if "Prepared Remarks" in full_text and "Questions" in full_text and "Answers" in full_text:
    parts = re.split(r"(Prepared Remarks\s*:|Questions (?:and|&) Answers\s*:)", full_text)
    # parts looks like: [intro, "Prepared Remarks:", remarks_text, "Questions & Answers:", qa_text]
    for i, part in enumerate(parts):
        if "Prepared Remarks" in part and i + 1 < len(parts):
            sections["prepared_remarks"] = parts[i + 1].strip()
        elif "Questions" in part and "Answers" in part and i + 1 < len(parts):
            sections["q_and_a"] = parts[i + 1].strip()
else:
    # If no clear markers found, treat everything as a single section
    sections["full_transcript"] = full_text

print("Sections found:", list(sections.keys()))
for name, text in sections.items():
    print(f"  {name}: {len(text):,} characters")

Sections found: ['prepared_remarks', 'q_and_a']
  prepared_remarks: 19,059 characters
  q_and_a: 29,906 characters


## Step 4: Parse speaker turns

Each speaker turn in the transcript has this format:

```
Timothy Donald Cook -- Chief Executive Officer

Good afternoon and thank you for joining us today...
```

We need to:
1. Detect lines matching the pattern `Name -- Role`
2. Capture all text until the next speaker
3. Return a list of dicts with `speaker`, `role`, and `text`

In [7]:
def parse_speaker_turns(text: str, section_name: str) -> list[dict]:
    """
    Parse a block of text and extract speaker turns.

    Args:
        text: The text of a section (prepared_remarks or q_and_a)
        section_name: "prepared_remarks" or "q_and_a"

    Returns:
        List of dicts with keys: speaker, role, section, text
    """
    turns = []

    parts = re.split(r'([A-Z][a-zA-Z .]+\n--\n[A-Za-z ,]+)', text)

    for i in range(1, len(parts), 2):
        name, role = parts[i].split("--", 1)
        dict_turn = {
            "speaker": name.strip(),
            "role": role.strip(),
            "section": section_name,
            "text": parts[i + 1].strip(),
        }
        if len(dict_turn["text"]) > 10:
            turns.append(dict_turn)

    return turns

### Test your implementation

Run the cell below to test. If it works, you should see the speakers with their roles.

In [8]:
# Test with real sections
all_turns = []
for section_name, section_text in sections.items():
    turns = parse_speaker_turns(section_text, section_name)
    all_turns.extend(turns)

print(f"Total speaker turns found: {len(all_turns)}")
print()

# Show summary
for i, turn in enumerate(all_turns[:5]):
    print(f"Turn {i+1}: {turn['speaker']} ({turn['role']}) [{turn['section']}]")
    print(f"  Text: {turn['text'][:100]}...")
    print()

Total speaker turns found: 75

Turn 1: Suhasini Chandramouli (Director, Investor Relations) [prepared_remarks]
  Text: Good afternoon, and welcome to the Apple Q1 fiscal year 2025 earnings conference call. My name is Su...

Turn 2: Timothy Donald Cook (Chief Executive Officer) [prepared_remarks]
  Text: Thank you, Suhasini. Good afternoon, everyone, and thanks for joining the call. Before I talk about ...

Turn 3: Kevan Parekh (Senior Vice President, Chief Financial Officer) [prepared_remarks]
  Text: Thanks, Tim, and good afternoon, everyone. I'm going to cover the results for the first quarter of o...

Turn 4: Suhasini Chandramouli (Director, Investor Relations) [prepared_remarks]
  Text: Thank you, Kevan. We ask that you limit yourself to two questions. Operator, may we have the first q...

Turn 5: Erik Woodring (Analyst) [q_and_a]
  Text: Great, guys. Thanks so much for taking my questions. You know, Tim, in your prepared remarks, you ha...



## Step 5: Save as structured JSON

We save the parsed transcript with all its metadata for use in the next notebooks.

In [14]:
transcript = {
    "company": COMPANY,
    "quarter": QUARTER,
    "date": DATE,
    "source_url": URL,
    "total_turns": len(all_turns),
    "speakers": list(set(t["speaker"] for t in all_turns)),
    "turns": all_turns,
}

# Save to disk
output_path = Path("../data/processed") / f"{COMPANY}_{QUARTER.replace('-', '_')}.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(transcript, f, indent=2, ensure_ascii=False)

print(f"Saved to: {output_path}")
print(f"  Company: {transcript['company']}")
print(f"  Quarter: {transcript['quarter']}")
print(f"  Speakers: {len(transcript['speakers'])}")
print(f"  Turns: {transcript['total_turns']}")

Saved to: ../data/processed/AAPL_Q1_2025.json
  Company: AAPL
  Quarter: Q1-2025
  Speakers: 15
  Turns: 75
